# Day 9: Transformer Architecture - Positional Encoding & Multi-Head Attention

## Introduction
Welcome to Day 9! Today, we transition from the high-level LLM fundamentals into the core architecture that powers modern AI: the **Transformer**. As an engineer, you know that understanding the underlying mechanisms of a system is crucial for debugging and optimization. 

We will focus on two foundational blocks:
1.  **Positional Encoding:** Injecting sequence order information into our data.
2.  **Multi-Head Attention:** Allowing the model to focus on different parts of the input sequence simultaneously.

## Core Theory (Just-in-Time)

### 1. The "Why" and "How" of Positional Encoding
**Why:** Traditional RNNs process tokens sequentially, inherently knowing which token came first. Transformers process all tokens simultaneously (in parallel). Without positional encoding, a Transformer would treat "The dog bit the man" and "The man bit the dog" as the exact same input. We must mathematically inject the *position* of each token into its embedding.

**How:** We add a unique positional vector to each token's embedding vector. The original paper ("Attention Is All You Need") uses sine and cosine functions of different frequencies. 
* Even dimensions get a sine wave.
* Odd dimensions get a cosine wave.
This continuous mathematical formulation allows the model to easily learn relative positions, even for sequences longer than those seen during training.

### 2. The "Why" and "How" of Multi-Head Attention
**Why:** Single attention calculates how much focus one token should give to all others based on *one* set of learned parameters. However, tokens relate to each other in multiple ways (e.g., grammatical relationship, semantic meaning, pronoun resolution). We need the model to capture multiple *types* of relationships simultaneously.

**How:** Instead of calculating attention once, we calculate it $h$ times in parallel ($h$ = number of "heads"). 
1. We project our input into $h$ different sets of Query (Q), Key (K), and Value (V) matrices.
2. We compute scaled dot-product attention for each head independently.
3. We concatenate the results from all heads.
4. We pass the concatenated output through a final linear projection to merge the insights.

### 3. AI Security & Production Implications
From a security and production standpoint, Transformers have inherent vulnerabilities based on their architecture:
- **Resource Exhaustion (OOM):** The multi-head attention mechanism scales quadratically $O(N^2)$ with the sequence length $N$. A maliciously long input (e.g., prompt injection packing thousands of tokens) can cause immediate Out-Of-Memory (OOM) errors, leading to a Denial of Service (DoS) for the application.
- **Context Window Overflows:** Always enforce strict validation bounds on the `seq_len` to drop or truncate inputs gracefully instead of letting the underlying attention mechanism fail unpredictably.


## Code Implementation: Basic

This Basic tier isolates the core concept of positional encoding with minimal boilerplate.

In [1]:
import math
from typing import List

def basic_positional_encoding(seq_len: int, d_model: int) -> List[List[float]]:
    """Generates basic positional encodings using sine/cosine."""
    pe = []
    for pos in range(seq_len):
        pos_embedding = []
        for i in range(d_model):
            exponent = (i // 2 * 2) / d_model
            denominator = math.pow(10000.0, exponent)
            value = pos / denominator
            
            if i % 2 == 0:
                pos_embedding.append(math.sin(value))
            else:
                pos_embedding.append(math.cos(value))
        pe.append(pos_embedding)
    return pe

# Usage
if __name__ == "__main__":
    pe_matrix = basic_positional_encoding(4, 4)
    print("Basic Positional Encoding (4x4):")
    for row in pe_matrix:
        print([f"{v:.4f}" for v in row])


Basic Positional Encoding (4x4):
['0.0000', '1.0000', '0.0000', '1.0000']
['0.8415', '0.5403', '0.0100', '1.0000']
['0.9093', '-0.4161', '0.0200', '0.9998']
['0.1411', '-0.9900', '0.0300', '0.9996']


## Code Implementation: Medium

The Medium tier emphasizes clean OOP principles and state management by representing an `AttentionHead` as a class.

In [2]:
import math
from typing import List

class AttentionHead:
    """An OOP representation of a single attention head."""
    def __init__(self, d_model: int, d_k: int):
        self.d_model = d_model
        self.d_k = d_k
        
        self.W_q = self._init_weights(d_model, d_k)
        self.W_k = self._init_weights(d_model, d_k)
        self.W_v = self._init_weights(d_model, d_k)

    def _init_weights(self, rows: int, cols: int) -> List[List[float]]:
        # Simple deterministic initialization for reproducibility
        return [[0.1 for _ in range(cols)] for _ in range(rows)]
        
    def _matrix_multiply(self, A: List[List[float]], B: List[List[float]]) -> List[List[float]]:
        m, n, p = len(A), len(A[0]), len(B[0])
        C = [[0.0 for _ in range(p)] for _ in range(m)]
        for i in range(m):
            for j in range(p):
                for k in range(n):
                    C[i][j] += A[i][k] * B[k][j]
        return C

    def forward(self, x: List[List[float]]) -> List[List[float]]:
        """Process input x (seq_len x d_model) through this head."""
        # 1. Linear Projections
        Q = self._matrix_multiply(x, self.W_q)
        K = self._matrix_multiply(x, self.W_k)
        V = self._matrix_multiply(x, self.W_v)
        
        # 2. Scaled Dot-Product Attention
        K_T = [[K[j][i] for j in range(len(K))] for i in range(len(K[0]))]
        scores = self._matrix_multiply(Q, K_T)
        
        scale = 1.0 / math.sqrt(self.d_k)
        for i in range(len(scores)):
            for j in range(len(scores[0])):
                scores[i][j] *= scale
                
        attention_weights = []
        for row in scores:
            max_val = max(row)
            exp_vals = [math.exp(val - max_val) for val in row]
            sum_exp = sum(exp_vals)
            attention_weights.append([val / sum_exp for val in exp_vals])
            
        output = self._matrix_multiply(attention_weights, V)
        return output

# Usage
if __name__ == "__main__":
    head = AttentionHead(d_model=4, d_k=2)
    dummy_x = [[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]]
    output = head.forward(dummy_x)
    print("\nMedium OOP Attention Head - Output:")
    for row in output:
         print([f"{v:.4f}" for v in row])



Medium OOP Attention Head - Output:
['2.4492', '2.4492']
['2.5956', '2.5956']


## Code Implementation: Advanced

The Advanced tier provides a production-grade multi-head attention implementation with strict type hinting, docstrings, error handling, input bounds validation (to mitigate DOS/OOM risks), and exact imports.

In [3]:
import math
import logging
from typing import List, Tuple, Optional

# Setup basic production logger
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

class MultiHeadAttention:
    """
    Production-grade Multi-Head Attention module.
    
    Includes strict input validation and limits to prevent resource exhaustion.
    """
    def __init__(self, d_model: int, num_heads: int, max_seq_len: int = 1024):
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be perfectly divisible by num_heads ({num_heads}).")
            
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.max_seq_len = max_seq_len
        
        logger.info(f"Initialized MultiHeadAttention: d_model={d_model}, num_heads={num_heads}, d_k={self.d_k}")

    def _validate_input(self, x: List[List[float]]) -> None:
        """Validates sequence length to prevent OOM DOS attacks."""
        seq_len = len(x)
        if seq_len > self.max_seq_len:
            error_msg = f"Input sequence length ({seq_len}) exceeds max allowed ({self.max_seq_len})."
            logger.error(error_msg)
            raise ValueError(error_msg)
            
        if not x:
            raise ValueError("Input sequence cannot be empty.")
            
        if len(x[0]) != self.d_model:
            raise ValueError(f"Input feature dimension ({len(x[0])}) does not match d_model ({self.d_model}).")

    def _scaled_dot_product_attention(
        self, Q: List[List[float]], K: List[List[float]], V: List[List[float]]
    ) -> List[List[float]]:
        """Core attention logic."""
        # Transpose K
        K_T = [[K[j][i] for j in range(len(K))] for i in range(len(K[0]))]
        
        # Q * K^T
        scores = [[sum(Q[i][k] * K_T[k][j] for k in range(len(K_T))) for j in range(len(K_T[0]))] for i in range(len(Q))]
        
        # Scale by 1/sqrt(d_k)
        scale = 1.0 / math.sqrt(self.d_k)
        scaled_scores = [[val * scale for val in row] for row in scores]
        
        # Softmax
        attention_weights = []
        for row in scaled_scores:
            max_val = max(row)
            exp_vals = [math.exp(val - max_val) for val in row]
            sum_exp = sum(exp_vals)
            attention_weights.append([val / sum_exp for val in exp_vals])
            
        # Output = Attention * V
        output = [[sum(attention_weights[i][k] * V[k][j] for k in range(len(V))) for j in range(len(V[0]))] for i in range(len(attention_weights))]
        return output

    def forward(self, x: List[List[float]]) -> List[List[float]]:
        """
        Simulates multi-head attention processing.
        
        Args:
            x: Input sequence of shape (seq_len, d_model)
            
        Returns:
            Output sequence of shape (seq_len, d_model)
        """
        self._validate_input(x)
        
        seq_len = len(x)
        head_outputs: List[List[List[float]]] = []
        
        for h in range(self.num_heads):
            start_idx = h * self.d_k
            end_idx = start_idx + self.d_k
            
            # Simulate slicing into heads for Q, K, V
            Q_head = [[row[i] for i in range(start_idx, end_idx)] for row in x]
            K_head = [[row[i] for i in range(start_idx, end_idx)] for row in x]
            V_head = [[row[i] for i in range(start_idx, end_idx)] for row in x]
            
            head_out = self._scaled_dot_product_attention(Q_head, K_head, V_head)
            head_outputs.append(head_out)
            
        # Concatenate heads horizontally
        final_output = []
        for r in range(seq_len):
            combined_row = []
            for head in head_outputs:
                combined_row.extend(head[r])
            final_output.append(combined_row)
            
        return final_output

# Usage
if __name__ == "__main__":
    try:
        mha = MultiHeadAttention(d_model=6, num_heads=2, max_seq_len=512)
        dummy_input = [
            [1.0, 0.5, 0.2, 0.1, 0.8, 0.9],
            [0.2, 0.9, 0.1, 0.5, 0.1, 0.4],
            [0.8, 0.1, 0.6, 0.9, 0.2, 0.3]
        ]
        output = mha.forward(dummy_input)
        print("\nAdvanced Output (3 tokens, 6 dimensions):")
        for row in output:
            print([f"{v:.4f}" for v in row])
    except Exception as e:
        logger.error(f"Execution failed: {e}")


INFO:__main__:Initialized MultiHeadAttention: d_model=6, num_heads=2, d_k=3



Advanced Output (3 tokens, 6 dimensions):
['0.7133', '0.4791', '0.3038', '0.4222', '0.4522', '0.6053']
['0.6396', '0.5415', '0.2729', '0.5079', '0.3669', '0.5305']
['0.7225', '0.4497', '0.3260', '0.5331', '0.3496', '0.5124']


## Common Pitfalls

1.  **Context Window Overflows:** Transformers process attention across the entire sequence length $N$. The memory and compute cost of self-attention scales quadratically, $O(N^2)$. Feeding excessive context without summarization or chunking will cause Out-Of-Memory (OOM) errors on your GPUs.
2.  **Dimension Mismatch Errors:** The most common bug when building custom transformer blocks is tensor shape mismatch. Always aggressively assert that `d_model % num_heads == 0`.
3.  **Ignoring Masking:** In the decoder portion of a transformer (used in models like GPT), you *must* apply a causal mask to the attention scores before the softmax step. If you forget this, the model "looks into the future" during training and fails catastrophically during inference.
4.  **Floating Point Instability:** Large values in the dot-product before softmax can cause exponential explosion. That is *why* we divide by $\sqrt{d_k}$. Always ensure your scaling factor is correctly applied when modifying attention logic.


## Practical Lab / Homework

**Your Task for Today:**

Below is a skeleton for a multi-head attention processing task. Complete the `concatenate_heads` function that takes a list of processed heads (each of shape `seq_len` x `d_k`) and horizontally merges them into a single matrix of shape `seq_len` x `(d_k * num_heads)`. 

*Constraint:* Do not use external libraries like NumPy. Pure Python only.
*Optional:* Record a 3-minute async video walkthrough explaining your concatenation logic.

In [4]:
from typing import List

def concatenate_heads(heads_list: List[List[List[float]]]) -> List[List[float]]:
    """
    Concatenates a list of attention heads horizontally.
    
    Args:
        heads_list: A list containing num_heads matrices. 
                    Each matrix is of shape (seq_len, d_k).
    Returns:
        A single matrix of shape (seq_len, d_k * num_heads).
    """
    if not heads_list:
        return []
        
    seq_len = len(heads_list[0])
    final_output = []
    
    for r in range(seq_len):
        combined_row = []
        for head in heads_list:
            combined_row.extend(head[r])
        final_output.append(combined_row)
        
    return final_output

# --- Lab Assertions ---
if __name__ == "__main__":
    # 2 heads. Each head processed 2 tokens with d_k=2
    head_1 = [
        [0.1, 0.2],
        [0.3, 0.4]
    ]
    head_2 = [
        [0.9, 0.8],
        [0.7, 0.6]
    ]
    
    result = concatenate_heads([head_1, head_2])
    
    print("\nLab Output - Concatenated Heads:")
    for row in result:
         print([f"{val: .4f}" for val in row])
         
    assert len(result) == 2, "Result should have 2 rows (seq_len)"
    assert len(result[0]) == 4, "Result should have 4 columns (2 heads * d_k 2)"
    assert result[0] == [0.1, 0.2, 0.9, 0.8], "Concatenation mismatch in row 1"
    print("\nSuccess! Concatenation works as expected.")



Lab Output - Concatenated Heads:
[' 0.1000', ' 0.2000', ' 0.9000', ' 0.8000']
[' 0.3000', ' 0.4000', ' 0.7000', ' 0.6000']

Success! Concatenation works as expected.


## Reference Links

1. [Attention Is All You Need (Original Paper)](https://arxiv.org/abs/1706.03762)
2. [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/)
3. [Transformers from Scratch (Peter Bloem)](http://peterbloem.nl/blog/transformers)